# L2 — Cross-Sectional + Dual Momentum sur le panier anti-biais

> **Ladder ML #1409** | L1 TSMOM (floor) · **L2 cross-sectional + dual momentum** · L3 Trend Long-Horizon · L4 Decision Transformer

**Issue** : [#4876](https://github.com/jsboige/CoursIA/issues/4876) (stub ladder discovery, tranche 2 de 3) · **Epic** : [#1409](https://github.com/jsboige/CoursIA/issues/1409) — Curriculum trading V3 (Long-Horizon Trend & Regime) · **Verdict source** : PR #1575 L2 cross-sectional + dual momentum NO BEATS, PR #1472 L1+L2 momentum baselines Wave 1.

L1 nous a appris qu'un signal momentum time-series est profitable **en brut** (confirme Moskowitz-Ooi-Pedersen 2012) mais que les coûts de transaction **(rebalancement journalier x 25 actifs)** le détruisent en net. L2 attaque ce probleme par **deux familles** :

1. **Cross-sectional momentum** : on classe les actifs par leur rendement passe (ranking cross-section), on long le top quantile (top-5 sur 25), short ou cash le bottom quantile. **Mensuel**, pas journalier -> frais ~10x moindres que L1.
2. **Dual momentum (Antonacci)** : on combine momentum **relative** (cross-section) + momentum **absolue** (vs 0). Si la moyenne des top quantile a un momentum absolu <= 0 -> on deplace en cash/obligations. C'est exactement le filtre anti-bear-market qui rend la strategie defense.

**Plan du notebook** :

1. Chargement du panier anti-biais (donnees cachees localement).
2. Baseline buy-and-hold equipondere (meme barre que L1, pour comparaison directe).
3. Cross-sectional momentum : top-5 long, bottom-5 short/cash, sweep sur 3 lookbacks.
4. Dual momentum : ajout du filtre absolu + cash.
5. Comparaison L2 vs L1 vs B&H, stress test 50 bps, exercises.


## 1. Chargement du panier anti-biais (donnees locales mises en cache)

On reutilise le **panier anti-biais 25 symboles / 7 classes d'actifs** (`PANIER_GROUPS`) charge depuis le cache local `datasets/panier/` (voir le module `panier_loader`, 79 fichiers CSV deterministes). **Aucun acces reseau** : `auto_fetch=False`.

In [ ]:
import importlib.util
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Charger le pipeline canonique scripts/L2_dual_momentum.py comme un module.
SCRIPTS = Path("scripts").resolve()
spec = importlib.util.spec_from_file_location("L2_dual_momentum", SCRIPTS / "L2_dual_momentum.py")
L2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(L2)

# Donnees cachees (pas de reseau).
from panier_loader import load_panier_closes, PANIER_GROUPS

closes = load_panier_closes(auto_fetch=False)
print(f"Panier : {closes.shape[1]} symboles x {closes.shape[0]:>5} jours")
print(f"Periode : {closes.index[0].date()} -> {closes.index[-1].date()}")
print(f"Groupes : { {k: len(v) for k, v in PANIER_GROUPS.items()} }")
print(f"Premiers symboles : {list(closes.columns[:6])}")


## 2. Baseline buy-and-hold equipondere

Meme baseline que L1 pour comparaison directe. **Sharpe cible L1 = 1.15** (constate en walk-forward 5-fold x 4 seeds). On cherche a battre cette barre avec **10x moins de frais** (rebalancement mensuel vs journalier).

In [ ]:
SEEDS = [0, 1, 7, 42]
N_SPLITS, GAP = 5, 21

bh_result = L2.run_buyhold_baseline(closes, SEEDS, N_SPLITS, GAP)
bh_mean = float(np.mean([s["sharpe"] for s in bh_result["seeds"]]))
bh_cagr = float(np.mean([s["cagr"] for s in bh_result["seeds"]]))
print(f"Buy-and-Hold equipondere (moyenne {len(SEEDS)} seeds, {N_SPLITS} folds):")
print(f"  Sharpe = {bh_mean:.4f}")
print(f"  CAGR   = {bh_cagr:.4f}")
print(f"  OOS bars = {bh_result['seeds'][0]['n_oos']}")
print(f"\nRappel L1 TSMOM (21j rebal journalier, floor non-ML):")
print(f"  Sharpe NET 21j ~ -2.56, 63j ~ -2.40, 126j ~ -2.26, 252j ~ -2.37")
print(f"\nNotre cible : battre 1.15 en NET avec rebalancement MENSUEL.")


## 3. Cross-Sectional Momentum : top-5 long, bottom-5 short/cash

**Hypothese** : un rebalancement **mensuel** (21j) reduit les frais d'un facteur ~21 par rapport a L1 journalier. **Lookbacks** : 63j, 126j, 252j (3 mois, 6 mois, 12 mois) - typiques pour le momentum factor.

Le script `run_cross_sectional(closes, lookback, seeds, n_splits, gap)` retourne un dict `{seeds: [...], lookback, top_n}` avec **Sharpe brut**, **Sharpe net** (apres couts equities/crypto), et **trades/seed**.

In [ ]:
LOOKBACKS_L2 = [63, 126, 252]
TOP_N = 5  # top-5 long, bottom-5 short ou cash

cs_results = []
for lb in LOOKBACKS_L2:
    res = L2.run_cross_sectional(closes, lb, SEEDS, N_SPLITS, GAP, top_n=TOP_N)
    cs_results.append(res)
    mean_gross = float(np.mean([s["gross_sharpe"] for s in res["seeds"]]))
    mean_net = float(np.mean([s["net_sharpe"] for s in res["seeds"]]))
    trades = int(np.mean([s["total_trades"] for s in res["seeds"]]))
    print(f"CS | lb {lb:>3}j | gross {mean_gross:>+7.3f} | net {mean_net:>+7.3f} | "
          f"~{trades:>5} trades/seed (mensuel !)")

print("\nNote : frais maintenant ~10x moindres que L1 (mensuel vs journalier).")


## 4. Dual Momentum (Antonacci) : combine CS + filtre absolu

**Innovation cle** : si la moyenne des rendements des top-N actifs sur le lookback est <= 0 (momentum absolu negatif), on **deplace 100% en cash**.

C'est la **regle anti-bear-market** d'Antonacci (2013, *Dual Momentum Investing*). Le payoff : pas d'exposition aux drawdowns equities majeurs (2008, 2020, 2022) -> Sharpe attendu plus stable.

Le script `run_dual_momentum(closes, lookback, seeds, n_splits, gap, top_n)` retourne la meme structure que CS plus une cle `cash_days` qui compte le nombre de jours passes en cash sur la periode OOS.

In [ ]:
dm_results = []
for lb in LOOKBACKS_L2:
    res = L2.run_dual_momentum(closes, lb, SEEDS, N_SPLITS, GAP, top_n=TOP_N)
    dm_results.append(res)
    mean_gross = float(np.mean([s["gross_sharpe"] for s in res["seeds"]]))
    mean_net = float(np.mean([s["net_sharpe"] for s in res["seeds"]]))
    cash_days = float(np.mean([s["cash_days"] for s in res["seeds"]]))
    trades = int(np.mean([s["total_trades"] for s in res["seeds"]]))
    print(f"DM | lb {lb:>3}j | gross {mean_gross:>+7.3f} | net {mean_net:>+7.3f} | "
          f"~{trades:>5} trades | {cash_days:>6.1f} jours en cash (~{cash_days/252*100:>4.1f}%)")

print("\nFiltre cash active : la proportion de temps en cash varie selon le regime.")


### Table de verdict canonique

Comparaison L2 (CS + DM) vs L1 (TSMOM journalier) vs B&H. **Verdict NO BEATS** signifie que la strategie ne bat **pas significativement** le B&H (delta NET Sharpe <= 2 sigma, ou signe negatif). **Verdict BEATS** = superieur avec confiance >= 2 sigma.

Le verdict est calcule par `compute_verdict(results, bh_sharpe)` du pipeline canonique (meme convention que L1 : `delta_sharpe >= 2 * std(deltas)`).

In [ ]:
rows = []
for strat_name, results in [("CS", cs_results), ("DM", dm_results)]:
    for res in results:
        v = L2.compute_verdict(res, bh_mean)
        lb = res["lookback"]
        mean_gross = float(np.mean([s["gross_sharpe"] for s in res["seeds"]]))
        mean_net = float(np.mean([s["net_sharpe"] for s in res["seeds"]]))
        cash_str = ""
        if strat_name == "DM":
            cash_days = float(np.mean([s.get("cash_days", 0) for s in res["seeds"]]))
            cash_str = f" | cash {cash_days/closes.shape[0]*100:>4.1f}%"
        rows.append({
            "strategy": strat_name,
            "lookback_j": lb,
            "gross_sharpe": round(mean_gross, 3),
            "net_sharpe": round(mean_net, 3),
            "B_and_H": round(bh_mean, 3),
            "delta_vs_BH": round(v["delta_sharpe"], 3),
            "t_stat": round(v["t_stat"], 2),
            "seeds_pos": f"{v['seeds_positive_delta']}/{v['n_seeds']}",
            "verdict": v["verdict"] + cash_str,
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print(f"Rappel L1 TSMOM (journalier, floor): TOUS les lookbacks NO BEATS en NET.")
print(f"L2 attendu : gains partiels seulement (CS profite du rebal mensuel, DM protege en bear).")


### Lecture du verdict

- **Cross-sectional momentum** : la version mensuelle long/short devrait ameliorer le L1 (frais divises par ~21). Les petits lookbacks (63j) captent mieux le signal reversal court-terme que les longs (252j).
- **Dual momentum** : le filtre absolu deplace en cash pendant les baissiers. C'est payant en theorie, mais uniquement si le signal cash n'est pas **systematiquement** declenche pendant les rallies (anti-pattern : sortir en 2023-09 aurait manque le rally Q4).
- **L1 vs L2** : L2 devrait avoir des **Sharpes nets moins desastreux** que L1, mais battre le B&H (1.15) reste difficile car : (a) le panier anti-biais est selectionne pour etre difficile (drift d'equities long terme), (b) les couts de swing long/short sur 25 actifs restent significatifs (5-15 bps selon classe), (c) le marche US 2010-2024 est un des plus forts ever, donc le B&H est une barre haute.


## 5. Stress test : couts x 5 (regime 50 bps)

En marche stressee (liquidite reduite, spreads elargis, market impact x10), les strategies a rebalancement frequent sont penalisees. L2 mensuelle devrait mieux resister que L1 journaliere. On relance le **meilleur lookback CS et DM** en mode stress (50 bps).

In [ ]:
best_cs_lb = max(cs_results, key=lambda r: float(np.mean(
    [s["net_sharpe"] for s in r["seeds"]])))["lookback"]
best_dm_lb = max(dm_results, key=lambda r: float(np.mean(
    [s["net_sharpe"] for s in r["seeds"]])))["lookback"]

cs_stress = L2.run_cross_sectional(closes, best_cs_lb, SEEDS, N_SPLITS, GAP, top_n=TOP_N, stress=True)
dm_stress = L2.run_dual_momentum(closes, best_dm_lb, SEEDS, N_SPLITS, GAP, top_n=TOP_N, stress=True)
cs_stress_net = float(np.mean([s["net_sharpe"] for s in cs_stress["seeds"]]))
dm_stress_net = float(np.mean([s["net_sharpe"] for s in dm_stress["seeds"]]))
print(f"Stress test (50 bps) sur le meilleur lookback CS ({best_cs_lb}j) :")
print(f"  CS net Sharpe (stress) = {cs_stress_net:+.4f}")
print(f"\nStress test (50 bps) sur le meilleur lookback DM ({best_dm_lb}j) :")
print(f"  DM net Sharpe (stress) = {dm_stress_net:+.4f}")
print(f"\nComparaison L1 TSMOM (stress 50 bps, 126j floor) = ~ -19.13 (cf docs/L1_tsmom.md)")
print(f"L2 devrait etre nettement meilleur en stress grace au rebal mensuel.")


## 6. Exercices

Les exercices sont a completer. Conformement a la regle pedagogique (C.1, user 2026-04-26), on utilise `result = None  # TODO etudiant` - le notebook s'execute de bout en bout meme avec les exercices non completes. Voir [exercise-example-labeling.md](../../../.claude/rules/exercise-example-labeling.md) pour la convention.

### Exercice 1 — Varier la taille du top-N

**Contexte** : le sweep canonique utilise top-5 long / bottom-5 short/cash sur 25 actifs. Or la sensibilite au top-N est connue : top-3 concentre, top-10 dilue. On veut voir si top-3 ou top-7 ameliore le verdict CS/DM en NET.

**Travail** : implementer `cs_dm_verdict_by_topn(top_ns, closes, lookback, seeds, n_splits, gap)` qui boucle sur top_ns=[3, 5, 7, 10] pour CS et DM, et retourne `{strategy: {top_n: verdict_dict}}`.

In [ ]:
def cs_dm_verdict_by_topn(top_ns, closes, lookback, seeds, n_splits, gap):
    # Retourne {strategy: {top_n: verdict_dict}} pour CS et DM.
    # strategy in ("CS", "DM").
    # TODO etudiant : boucler, run_cross_sectional / run_dual_momentum + compute_verdict
    result = None  # TODO etudiant
    return result

alt_topns = [3, 5, 7, 10]
res = cs_dm_verdict_by_topn(alt_topns, closes, 126, SEEDS, N_SPLITS, GAP)
if res is not None:
    for strat in ("CS", "DM"):
        for tn, v in res[strat].items():
            print(f"{strat} top-{tn:>2}: {v['verdict']} (delta={v['delta_sharpe']:+.3f})")


### Exercice 2 — Seuil de momentum absolu pour DM

**Contexte** : le filtre DM par defaut declenche le cash quand la moyenne du top-N est <= 0. Mais le seuil pourrait etre tighte (top-N <= 0) ou relax (top-N <= +5% annualise). Quel seuil maximise le Sharpe NET sur walk-forward ?

**Travail** : modifier le pipeline pour exposer un parametre `abs_threshold` et comparer {0.0, 0.02, 0.05, 0.10} en regardant le Sharpe NET et la frequence cash.

In [ ]:
def dm_abs_threshold_scan(thresholds, closes, lookback, seeds, n_splits, gap, top_n):
    # Retourne {threshold: (net_sharpe, cash_fraction)} pour chaque seuil.
    # TODO etudiant : utiliser run_dual_momentum ou wrapper avec param abs_threshold
    result = None  # TODO etudiant
    return result

thresholds = [0.0, 0.02, 0.05, 0.10]
res = dm_abs_threshold_scan(thresholds, closes, 126, SEEDS, N_SPLITS, GAP, TOP_N)
if res is not None:
    for t, (s, c) in res.items():
        print(f"DM abs_thr {t:>4.2f}: net Sharpe {s:+.3f}, cash {c*100:>4.1f}%")


### Exercice 3 — Combien de jours en cash sauve le drawdown ?

**Contexte** : DM protege en bear via cash. Mais **combien** de drawdown le cash sauve-t-il reellement ? Sans L2 DM, le B&H ou CS aurait eu un drawdown X ; avec DM il a un drawdown Y. L'ecart Y-X est le **drawdown protege** par le filtre cash.

**Travail** : pour chaque seed, calculer le max drawdown relatif de B&H, CS, DM sur la periode OOS. Retourner `{strategy: {seed: max_dd}}` et la moyenne par strategie.

In [ ]:
def max_drawdown_by_strategy(strategies_returns, seeds):
    # strategies_returns : dict[strategy -> dict[seed -> pd.Series de rendements]]
    # Retourne {strategy: {seed: max_drawdown}} (positif = drawdown).
    # TODO etudiant : pour chaque (strategy, seed), calculer le max drawdown
    # (running peak - value) / running peak du cumul des rendements.
    result = None  # TODO etudiant
    return result

# Construction des strategies_returns depuis les resultats CS / DM / B&H.
# TODO etudiant : extraire les series de rendements OOS par seed depuis cs_results,
# dm_results, et bh_result, puis appliquer max_drawdown_by_strategy.
strategies_returns = None  # TODO etudiant
dd = max_drawdown_by_strategy(strategies_returns, SEEDS)
if dd is not None:
    for strat, seed_dd in dd.items():
        mean_dd = float(np.mean(list(seed_dd.values())))
        print(f"{strat}: max DD moyen = {mean_dd*100:>5.2f}%")


## 7. Conclusion

| Idee cle | Traduction |
|---|---|
| L1 = floor non-ML | TSMOM journalier detruit en net (Sharpe ~ -2.6). |
| L2 CS = cross-sectional ranking | Long top-N / short bottom-N avec rebal **mensuel** divise les frais par ~21 vs L1. |
| L2 DM = filtre absolu (Antonacci) | Combine ranking relatif + momentum absolu, deplace en cash quand top-N <= 0. |
| Verdict attendu | CS et DM ameliorent le L1 (frais moindres) mais battre B&H 1.15 reste difficile. |
| Lecon pour L4+ | Le marche US 2010-2024 est un regime anomalique (B&H tres fort). L2 prepare le terrain pour L4 Decision Transformer (qui ajoute du context conditioning). |

**Note editoriale** : les veritables resultats chiffres dependent de l'execution du pipeline canonique (`scripts/L2_dual_momentum.py`). Le verdict **NO BEATS** documente dans la PR historique #1575 (Wave 1) peut evoluer si le pipeline est re-parametre (top-N, lookback, stress).